# HeatGuard AI — Phase 3: Baseline Models & Evaluation Engine

**Objective:** Train, benchmark, and serialize the 3 foundational baseline models on the 74-year dataset (`data/model_ready_dataset.csv`):
1. **Baseline 1: Climatological & Persistence Heuristic** (Physical Domain Anchor)
2. **Baseline 2: Balanced Logistic Regression (L2)** (Linear Convex Benchmark)
3. **Baseline 3: Tree Ensemble Benchmark** (LightGBM Balanced Forest)

**Key Protocols:**
- Chronological evaluation: Train (<= 2015), Val (2016-2020), Test (2021-2024).
- Decision threshold $\tau^*$ tuned strictly on Validation set to maximize F1-score, then applied to Test set.
- Primary optimization metric: **PR-AUC (Precision-Recall AUC)**.

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb

# Ensure project root is in path
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from models.evaluate import compute_metrics, find_optimal_threshold, save_metrics_summary
from models.train_baselines import PersistenceBaseline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print(f"Project Root: {REPO_ROOT}")

## 1. Load Model-Ready Dataset and Feature Schema

In [ ]:
data_path = REPO_ROOT / "data" / "model_ready_dataset.csv"
schema_path = REPO_ROOT / "data" / "feature_schema.json"

print(f"Reading data from: {data_path}")
df = pd.read_csv(data_path)
with open(schema_path, "r", encoding="utf-8") as f:
    schema = json.load(f)

feature_cols = schema["feature_columns"]
target_col = schema["primary_target"]

print(f"Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Total input features: {len(feature_cols)}")
print(f"Primary target: {target_col}")

## 2. Chronological Split Separation (Train / Val / Test)

In [ ]:
splits = {}
for s in ("train", "val", "test"):
    sub = df[df["split"] == s]
    X = sub[feature_cols].copy()
    y = sub[target_col].to_numpy().astype(int)
    splits[s] = (X, y)
    print(f"Split {s.upper():<5}: {len(sub):>7,} rows | {int(y.sum()):>5} Heatwave days ({y.mean()*100:.3f}% prevalence)")

X_train, y_train = splits["train"]
X_val, y_val = splits["val"]
X_test, y_test = splits["test"]

## 3. Baseline 1: Climatological Persistence Heuristic

In [ ]:
results = {}

persistence = PersistenceBaseline()
persistence.fit(X_train, y_train)

train_probs_p = persistence.predict_proba(X_train)[:, 1]
val_probs_p = persistence.predict_proba(X_val)[:, 1]
test_probs_p = persistence.predict_proba(X_test)[:, 1]

# Find optimal threshold on validation set
p_thresh, p_f1 = find_optimal_threshold(y_val, val_probs_p)
print(f"Persistence Optimal Threshold (Val): tau = {p_thresh:.3f}, Val F1 = {p_f1:.4f}")

results["Persistence_Heuristic"] = {
    "train": compute_metrics(y_train, train_probs_p, threshold=p_thresh),
    "val": compute_metrics(y_val, val_probs_p, threshold=p_thresh),
    "test": compute_metrics(y_test, test_probs_p, threshold=p_thresh),
    "optimal_threshold": p_thresh
}

print("\n--- Persistence Baseline Test Metrics ---")
for k, v in results["Persistence_Heuristic"]["test"].items():
    print(f"  {k:25s}: {v}")

## 4. Baseline 2: Balanced Logistic Regression (L2 Regularized)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

t0 = time.time()
logreg = LogisticRegression(
    C=0.1,
    penalty="l2",
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
    solver="lbfgs"
)
logreg.fit(X_train_scaled, y_train)
t_lr = time.time() - t0
print(f"Logistic Regression fitted in {t_lr:.2f} seconds.")

train_probs_lr = logreg.predict_proba(X_train_scaled)[:, 1]
val_probs_lr = logreg.predict_proba(X_val_scaled)[:, 1]
test_probs_lr = logreg.predict_proba(X_test_scaled)[:, 1]

# Tune threshold on validation set
lr_thresh, lr_f1 = find_optimal_threshold(y_val, val_probs_lr)
print(f"Logistic Regression Optimal Threshold (Val): tau = {lr_thresh:.3f}, Val F1 = {lr_f1:.4f}")

results["Logistic_Regression"] = {
    "train": compute_metrics(y_train, train_probs_lr, threshold=lr_thresh),
    "val": compute_metrics(y_val, val_probs_lr, threshold=lr_thresh),
    "test": compute_metrics(y_test, test_probs_lr, threshold=lr_thresh),
    "optimal_threshold": lr_thresh,
    "training_time_seconds": round(t_lr, 2)
}

print("\n--- Logistic Regression Test Metrics ---")
for k, v in results["Logistic_Regression"]["test"].items():
    print(f"  {k:25s}: {v}")

### Top Feature Coefficients in Logistic Regression

In [ ]:
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": logreg.coef_[0]
}).sort_values(by="coefficient", ascending=False)

print("Top 5 Positive Drivers of Heatwave Risk:")
print(coef_df.head(5).to_string(index=False))

print("\nTop 5 Negative Drivers:")
print(coef_df.tail(5).to_string(index=False))

## 5. Baseline 3: Tree Ensemble Benchmark (LightGBM Forest)

In [ ]:
t0 = time.time()
rf = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=10,
    learning_rate=0.05,
    num_leaves=31,
    class_weight="balanced",
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
rf.fit(X_train, y_train)
t_rf = time.time() - t0
print(f"Tree Ensemble fitted in {t_rf:.2f} seconds.")

train_probs_rf = rf.predict_proba(X_train)[:, 1]
val_probs_rf = rf.predict_proba(X_val)[:, 1]
test_probs_rf = rf.predict_proba(X_test)[:, 1]

# Tune threshold on validation set
rf_thresh, rf_f1 = find_optimal_threshold(y_val, val_probs_rf)
print(f"Tree Ensemble Optimal Threshold (Val): tau = {rf_thresh:.3f}, Val F1 = {rf_f1:.4f}")

results["Tree_Ensemble_Baseline"] = {
    "train": compute_metrics(y_train, train_probs_rf, threshold=rf_thresh),
    "val": compute_metrics(y_val, val_probs_rf, threshold=rf_thresh),
    "test": compute_metrics(y_test, test_probs_rf, threshold=rf_thresh),
    "optimal_threshold": rf_thresh,
    "training_time_seconds": round(t_rf, 2)
}

print("\n--- Tree Ensemble Test Metrics ---")
for k, v in results["Tree_Ensemble_Baseline"]["test"].items():
    print(f"  {k:25s}: {v}")

### Tree Ensemble Top Feature Importances

In [ ]:
rf_imp = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf.feature_importances_
}).sort_values(by="importance", ascending=False)

print("Top 10 Feature Importances (Tree Ensemble):")
print(rf_imp.head(10).to_string(index=False))

## 6. Comprehensive Benchmark Comparison Table

In [ ]:
rows = []
for model_name, data in results.items():
    for split in ("train", "val", "test"):
        m = data[split]
        rows.append({
            "Model": model_name,
            "Split": split.upper(),
            "PR-AUC": m["pr_auc"],
            "ROC-AUC": m["roc_auc"],
            "F1 (HW)": m["f1_heatwave"],
            "Precision": m["precision"],
            "Recall": m["recall"],
            "Brier Score": m["brier_score"],
            "Threshold": f"tau = {m['threshold']}"
        })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

## 7. Serialize Models & Export Benchmark Metrics

In [ ]:
save_dir = REPO_ROOT / "models" / "saved"
save_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(persistence, save_dir / "persistence_baseline.pkl")
joblib.dump(logreg, save_dir / "logreg_baseline.pkl")
joblib.dump(rf, save_dir / "rf_baseline.pkl")
joblib.dump(scaler, save_dir / "scaler.pkl")

json_path = REPO_ROOT / "data" / "baseline_metrics.json"
md_path = REPO_ROOT / "data" / "baseline_report.md"
save_metrics_summary(results, json_path, md_path)

print(f"All artifacts successfully serialized to: {save_dir}")
print(f"Metrics exported to: {json_path} and {md_path}")